# Suite2p — Mesoscope (MROI) Pipeline (suite2p 1.0: `settings` + `db`)

Run suite2p on a ScanImage **multi-ROI (mesoscope)** acquisition.

1. Load your **settings preset** (JSON written by the pipeline notebook, or a `settings.npy`) and deep-merge it onto `default_settings()`.
2. Build the dataset description `db` (paths, …).
3. Load the `ops.json` written by the MATLAB helper [`mesoscope_json_from_scanimage.m`](https://github.com/MouseLand/suite2p/blob/master/helpers/mesoscope_json_from_scanimage.m) and take **only** the recording-specific fields from it:
   - into **`db`** (they describe the data): `nplanes`, `nrois`, `lines`, `dx`, `dy`
   - into **`settings`**: `fs`
   - `mesoscan` is only used as a check — 1.0 has no mesoscan flag; multi-ROI data is described by `nrois`, `lines`, `dx`, `dy`.
4. Run `run_s2p(db=db, settings=settings)` and save the settings used as JSON.

In [ ]:
import suite2p
from suite2p import run_s2p, default_settings, default_db
import numpy as np
import json
from pathlib import Path

import copy

def deep_update(base, new):
    """Recursively merge `new` into `base` (nested dicts are merged, not replaced).
    run_s2p() only does a shallow merge, so a partial group such as
    {"registration": {"nonrigid": False}} would otherwise wipe the other
    registration defaults."""
    for k, v in new.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            deep_update(base[k], v)
        else:
            base[k] = v
    return base

def find_unknown_keys(d, ref, prefix=""):
    """Keys in `d` that do not exist in the reference dict (typos / removed keys)."""
    bad = []
    for k, v in d.items():
        if k not in ref:
            bad.append(prefix + k)
        elif isinstance(v, dict) and isinstance(ref[k], dict):
            bad += find_unknown_keys(v, ref[k], prefix + k + ".")
    return bad

def to_jsonable(v):
    if isinstance(v, dict):            return {k: to_jsonable(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):   return [to_jsonable(x) for x in v]
    if isinstance(v, np.ndarray):      return v.tolist()
    if isinstance(v, np.generic):      return v.item()
    if isinstance(v, Path):            return str(v)
    return v

def load_settings_file(path):
    """Load a settings file: a .json written by these notebooks, or a settings.npy
    written by suite2p into any suite2p/ or suite2p/planeN/ folder."""
    path = Path(path)
    assert path.exists(), f"Settings file not found: {path}"
    if path.suffix == ".json":
        with open(path) as f:
            return json.load(f)
    if path.suffix == ".npy":
        return np.load(path, allow_pickle=True).item()
    raise ValueError(f"Unsupported format: {path.suffix}")

print(f"suite2p version: {suite2p.__version__}")

## 1. Load your settings preset

In [ ]:
SETTINGS_FILE = r"/path/to/Dendrites_iGluSNFR4s.json"   # or settings.npy from any suite2p results folder

settings = deep_update(default_settings(), load_settings_file(SETTINGS_FILE))

unknown = find_unknown_keys(settings, default_settings())
if unknown:
    print(f"⚠️  Keys not in default_settings() (typos or pre-1.0 keys?): {unknown}")
print(f"Loaded settings from: {Path(SETTINGS_FILE).name}")

## 2. Load the MATLAB-generated MROI JSON

Only these seven fields are taken from it:

| key | goes to | meaning |
|---|---|---|
| `fs` | `settings["fs"]` | frame rate (Hz) |
| `nplanes` | `db["nplanes"]` | number of z-planes |
| `nrois` | `db["nrois"]` | number of ScanImage ROIs |
| `mesoscan` | (check only) | 1 = multi-ROI recording |
| `lines` | `db["lines"]` | per ROI, the row indices of the raw (interleaved) frame belonging to that ROI |
| `dx`, `dy` | `db["dx"]`, `db["dy"]` | per ROI, x / y pixel offset of the ROI in the stitched field of view |

In [ ]:
MESO_JSON = r"/path/to/session/ops.json"   # written by mesoscope_json_from_scanimage.m

assert Path(MESO_JSON).exists(), f"MROI JSON not found: {MESO_JSON}"
with open(MESO_JSON) as f:
    meso = json.load(f)

MESO_KEYS = ["fs", "nplanes", "nrois", "mesoscan", "lines", "dx", "dy"]
missing = [k for k in MESO_KEYS if k not in meso]
assert not missing, f"Missing keys in {MESO_JSON}: {missing}"
assert meso["mesoscan"], "mesoscan is 0 — this JSON does not describe a multi-ROI recording"

print("Fields taken from the MROI JSON:")
for k in MESO_KEYS:
    v = meso[k]
    print(f"  {k:9s} = {'%d ROI row-arrays, lengths %s' % (len(v), [len(r) for r in v]) if k == 'lines' else v}")

## 3. Build `db` and overwrite the MROI-specific fields

`data_path` and `save_path0` are set here, per session (they are not taken from the MATLAB JSON).

In [ ]:
db = default_db()
db["data_path"]  = [r"/path/to/session/tiffs"]
db["save_path0"] = r"/path/to/session/results"
# db["fast_disk"] = r"/scratch"

# ── MROI-specific fields from the MATLAB JSON ──
db["nplanes"] = int(meso["nplanes"])
db["nrois"]   = int(meso["nrois"])
db["lines"]   = [[int(i) for i in rows] for rows in meso["lines"]]
db["dx"]      = [int(v) for v in np.atleast_1d(meso["dx"])]
db["dy"]      = [int(v) for v in np.atleast_1d(meso["dy"])]
settings["fs"] = float(meso["fs"])

for p in db["data_path"]:
    assert Path(p).exists(), f"data_path not found: {p}"
Path(db["save_path0"]).mkdir(parents=True, exist_ok=True)

bad = find_unknown_keys(db, default_db())
assert not bad, f"Unknown db keys: {bad}"
print("db and settings ready")

## 4. Sanity checks

In [ ]:
nrois, lines = db["nrois"], db["lines"]
assert len(lines) == len(db["dx"]) == len(db["dy"]) == nrois, \
    f"nrois={nrois}, len(lines)={len(lines)}, len(dx)={len(db['dx'])}, len(dy)={len(db['dy'])}"

print(f"nrois={nrois}  nplanes={db['nplanes']}  fs={settings['fs']} Hz\n")
print("ROI layout (rows in raw frame -> offset in stitched FoV):")
for i, rows in enumerate(lines):
    print(f"  ROI {i}: rows {rows[0]}-{rows[-1]} ({len(rows)} rows)   dy={db['dy'][i]}  dx={db['dx'][i]}")

## 5. Check the memory-sensitive settings before running

MROI frames are large. With `registration.nonrigid = True`, registration VRAM grows with `registration.batch_size` and the number of non-rigid blocks. On a 12 GB GPU, ~30 was empirically safe for 1000×2000 frames (see the notes in `suite2p_pipeline_settings.ipynb`). `detection.nbins` sets the RAM used during detection.

In [ ]:
r = settings["registration"]
print(f"registration.batch_size = {r['batch_size']}   nonrigid = {r['nonrigid']}   block_size = {r['block_size']}")
print(f"extraction.batch_size   = {settings['extraction']['batch_size']}")
print(f"detection.nbins         = {settings['detection']['nbins']}   denoise = {settings['detection']['denoise']}")

## 6. Run the pipeline

In [ ]:
run_s2p(db=db, settings=settings)

save_dir = Path(db["save_path0"]) / db["save_folder"]
print("suite2p finished →", save_dir)

# Save the exact settings and db used
with open(save_dir / "settings_used.json", "w") as f:
    json.dump(to_jsonable(settings), f, indent=2)
with open(save_dir / "db_used.json", "w") as f:
    json.dump(to_jsonable(db), f, indent=2)

## 7. Inspect results

For multi-ROI data suite2p writes one `planeN` folder per ROI × plane (and a `combined` folder when more than one plane is processed and `settings["io"]["combined"]` is on).

In [ ]:
import matplotlib.pyplot as plt

print("Folders:", sorted(d.name for d in save_dir.iterdir() if d.is_dir()))
plane_dirs = sorted(save_dir.glob("plane*"))

# Pick a folder to look at: the combined one if it exists, else the first plane
result_dir = save_dir / "combined" if (save_dir / "combined").exists() else plane_dirs[0]
print("Loading from:", result_dir.name)

F      = np.load(result_dir / "F.npy",      allow_pickle=True)
Fneu   = np.load(result_dir / "Fneu.npy",   allow_pickle=True)
iscell = np.load(result_dir / "iscell.npy", allow_pickle=True)
cell_idx = np.where(iscell[:, 0].astype(bool))[0]

# Images and per-folder settings (fall back to the first plane if the folder has no npy copies)
src = result_dir if (result_dir / "reg_outputs.npy").exists() else plane_dirs[0]
reg_outputs  = np.load(src / "reg_outputs.npy", allow_pickle=True).item()
settings_out = np.load(src / "settings.npy",    allow_pickle=True).item()
fs = settings_out["fs"]

print(f"ROIs: {F.shape[0]}   cells: {len(cell_idx)}   frames: {F.shape[1]}   duration: {F.shape[1]/fs:.1f} s")

img = reg_outputs["meanImg"]
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(img, cmap="gray", vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
ax.set_title(f"Mean image ({src.name})"); ax.axis("off")
plt.show()

In [ ]:
n = min(20, len(cell_idx))
sel = np.random.default_rng(0).choice(cell_idx, size=n, replace=False)
Fc  = F[sel] - settings_out["extraction"]["neuropil_coefficient"] * Fneu[sel]
F0  = np.percentile(Fc, 8, axis=1, keepdims=True)
dFF = (Fc - F0) / (F0 + 1)
t   = np.arange(F.shape[1]) / fs

fig, ax = plt.subplots(figsize=(16, n * 0.5 + 1))
for i, tr in enumerate(dFF):
    ax.plot(t, tr + i * 3, lw=0.6, color="k")
ax.set_xlabel("Time (s)"); ax.set_yticks([]); ax.set_title(f"dF/F ({n} cells)")
plt.show()